# POC AutoGluon Tabular - Prédiction de la volatilité du Bitcoin

## Projet VolatiChainXplorerAI - Sprint 2

Ce notebook est consacré à l'expérimentation et à l'évaluation d'**AutoGluon Tabular** pour la prédiction de la volatilité du Bitcoin à **7 et 30 jours**.

L'objectif est d'évaluer les performances prédictives d'AutoGluon et de les comparer à celles obtenues avec PyCaret, dans le respect du protocole expérimental commun préalablement défini.

Les expérimentations s'appuient sur les **partitions chronologiques d'entraînement et de test déjà préparées, validées, figées et sauvegardées localement**. Ces partitions sont réutilisées à l'identique, sans nouveau découpage ni modification, afin de garantir la comparabilité et la reproductibilité des résultats.

### Démarche expérimentale

1. Charger les partitions d'entraînement et de test préalablement sauvegardées pour les horizons 7 et 30 jours.
2. Vérifier leur intégrité et leur conformité au protocole commun.
3. Configurer et entraîner les modèles avec AutoGluon Tabular.
4. Évaluer les performances à l'aide des métriques MAE, RMSE et R².
5. Comparer les résultats à ceux du modèle de référence `DummyRegressor` et de PyCaret.
6. Analyser les performances, les temps d'exécution et les ressources mobilisées.
7. Identifier les modèles les plus pertinents pour chaque horizon de prédiction.

### Finalité

Déterminer si AutoGluon constitue une solution AutoML adaptée aux exigences prédictives et aux contraintes techniques de VolatiChainXplorerAI.

Les résultats alimenteront le benchmark comparatif et contribueront à la justification du choix technologique dans le livrable E2 de certification.

## 1.Vérifier l’environnement

**Justification du choix de l'environnement AutoGluon**

AutoGluon Tabular 1.6.3 a été retenu comme version stable compatible avec Python 3.12 au moment de l'expérimentation. Un environnement virtuel dédié a été configuré afin d'isoler les dépendances et d'éviter les conflits avec PyCaret.

La version d'AutoGluon a été figée pour favoriser la reproductibilité du POC.

**Source :** [Documentation officielle AutoGluon](https://auto.gluon.ai/dev/install.html)

https://pypi.org/project/autogluon.tabular/1.6.3/

In [5]:
import platform
import sys

print("Version Python :", sys.version)
print("Exécutable Python :", sys.executable)
print("Système :", platform.platform())

Version Python : 3.12.9 (tags/v3.12.9:fdb8142, Feb  4 2025, 15:27:58) [MSC v.1942 64 bit (AMD64)]
Exécutable Python : d:\venvs\autogluon\Scripts\python.exe
Système : Windows-11-10.0.26100-SP0


## 2.Préparer les données du poc

In [ ]:
# Charger les librairies nécessaires
import pandas as pd
import random
import numpy as np

from pathlib import Path
from autogluon.tabular import TabularPredictor

In [25]:
# Définir le chemin du projet
project_root = Path().cwd().parent
sys.path.insert(0, str(project_root))

sets_dir = (
    project_root 
    / "data" 
    / "exports"
    / "datasets"
    / "benchmark_splits"
)

outputs_dir_7d = (
    project_root
    / "data"
    / "exports"
    / "autogluon_models"
    / "predictor_7d"
    
)
outputs_dir_7d.mkdir(parents=True, exist_ok=True)

outputs_dir_30d = (
    project_root
    / "data"
    / "exports"
    / "autogluon_models"
    / "predictor_30d"
    
)
outputs_dir_30d.mkdir(parents=True, exist_ok=True)

print("Chemin du projet :", project_root)
print("Chemin des jeux de données :", sets_dir)
print("Chemin des modèles AutoGluon 7d :", outputs_dir_7d)
print("Chemin des modèles AutoGluon 30d :", outputs_dir_30d)

Chemin du projet : d:\ProjectFolderDevAI_2025-2026\Certification BLOCS 2&3\Projet-VolatiChainXplorerAI
Chemin des jeux de données : d:\ProjectFolderDevAI_2025-2026\Certification BLOCS 2&3\Projet-VolatiChainXplorerAI\data\exports\datasets\benchmark_splits
Chemin des modèles AutoGluon 7d : d:\ProjectFolderDevAI_2025-2026\Certification BLOCS 2&3\Projet-VolatiChainXplorerAI\data\exports\autogluon_models\predictor_7d
Chemin des modèles AutoGluon 30d : d:\ProjectFolderDevAI_2025-2026\Certification BLOCS 2&3\Projet-VolatiChainXplorerAI\data\exports\autogluon_models\predictor_30d


In [34]:
# Charger les jeux de données du poc
dataset_names = [
    "train_7d",
    "validation_7d",
    "test_7d",
    "train_30d",
    "validation_30d",  
    "test_30d"
]

datasets = {}

for dataset_name in dataset_names:
    filename = sets_dir / f"{dataset_name}.csv"
    datasets[dataset_name] = pd.read_csv(
        filename,
        parse_dates=True
    )

    print(
        f"{dataset_name}.csv est chargées.",
        datasets[dataset_name].shape        
    )

train_7d.csv est chargées. (4228, 17)
validation_7d.csv est chargées. (724, 17)
test_7d.csv est chargées. (590, 17)
train_30d.csv est chargées. (4182, 17)
validation_30d.csv est chargées. (701, 17)
test_30d.csv est chargées. (590, 17)


## 3. Identification des variables et des cibles

Cette étape vérifie les colonnes présentes dans les datasets et identifie la cible propre à chaque horizon.

In [35]:
train_7d = datasets["train_7d"]
validation_7d = datasets["validation_7d"]
test_7d = datasets["test_7d"]

train_30d = datasets["train_30d"]
validation_30d = datasets["validation_30d"]
test_30d = datasets["test_30d"]

print("Liste des colonnes des sets à 7 jours")
print(train_7d.columns.tolist())

print("\nListe des colonnes des sets à 30 jours")
print(train_30d.columns.tolist())

# Recuperation des variables cibles 
column_only_7d = set(train_7d.columns) - set(train_30d.columns)
column_only_30d = set(train_30d.columns) - set(train_7d.columns)

print("\nColonne cible à 7 jours :", column_only_7d)
print("Colonne cible à  30 jours :", column_only_30d)


Liste des colonnes des sets à 7 jours
['date', 'log_return', 'volatility_3d', 'volatility_7d', 'volatility_14d', 'volatility_30d', 'daily_range', 'volume_change', 'market_cap_change', 'rate_mro', 'inflation_rate', 'unemployment_rate', 'monetary_m3_rate', 'log_return_lag_1d', 'log_return_lag_7d', 'log_return_lag_30d', 'target_volatility_7d']

Liste des colonnes des sets à 30 jours
['date', 'log_return', 'volatility_3d', 'volatility_7d', 'volatility_14d', 'volatility_30d', 'daily_range', 'volume_change', 'market_cap_change', 'rate_mro', 'inflation_rate', 'unemployment_rate', 'monetary_m3_rate', 'log_return_lag_1d', 'log_return_lag_7d', 'log_return_lag_30d', 'target_volatility_30d']

Colonne cible à 7 jours : {'target_volatility_7d'}
Colonne cible à  30 jours : {'target_volatility_30d'}


## 4.Préparer les données pour AutoGluon tabular

Nous créons ici les copie des colonne du dataframe sans la colonne "date". 

Le but étant de ne pas modifier les dataframe d'origine pour descontrôles chronologiques.

In [36]:
target_7d = "target_volatility_7d"
target_30d = "target_volatility_30d"

features_7d = [
    column
    for column in train_7d.columns
    if column not in ["date", target_7d]
]

features_30d = [
    column
    for column in train_30d.columns
    if column not in ["date", target_30d]
]

assert features_7d == features_30d, (
    "Les variables exlplicatives êtres identiques,"
    "mais elles sont differentes les unes des autres"
)

train_7d_model = train_7d.drop(columns="date").copy()
validation_7d_model = validation_7d.drop(columns="date").copy()
test_7d_model = test_7d.drop(columns="date").copy()

train_30d_model = train_30d.drop(columns="date").copy()
validation_30d_model = validation_30d.drop(columns="date").copy()
test_30d_model = test_30d.drop(columns="date").copy()

print(features_7d)
print(features_30d)
print("Nombre de variables explicatives :", len(features_7d))
print("train à 7 jours :", train_7d_model.shape)
print("validation à 7 jours :", validation_7d_model.shape)
print("train à 30 jours :", train_30d_model.shape)
print("validation à 30 jours :", validation_30d_model.shape)
print("test à 7 jours :", test_7d_model.shape)
print("test à 30 jours :", test_30d_model.shape)

['log_return', 'volatility_3d', 'volatility_7d', 'volatility_14d', 'volatility_30d', 'daily_range', 'volume_change', 'market_cap_change', 'rate_mro', 'inflation_rate', 'unemployment_rate', 'monetary_m3_rate', 'log_return_lag_1d', 'log_return_lag_7d', 'log_return_lag_30d']
['log_return', 'volatility_3d', 'volatility_7d', 'volatility_14d', 'volatility_30d', 'daily_range', 'volume_change', 'market_cap_change', 'rate_mro', 'inflation_rate', 'unemployment_rate', 'monetary_m3_rate', 'log_return_lag_1d', 'log_return_lag_7d', 'log_return_lag_30d']
Nombre de variables explicatives : 15
train à 7 jours : (4228, 16)
validation à 7 jours : (724, 16)
train à 30 jours : (4182, 16)
validation à 30 jours : (701, 16)
test à 7 jours : (590, 16)
test à 30 jours : (590, 16)


## 5. Configuration de l’expérience AutoGluon tabular à 7 et 30 jours

### A. Configuration de l'expérience AutoGluon à 7 jours

L'expérience utilise exclusivement la partition d'entraînement préalablement figée pour l'apprentissage des modèles.

La partition de validation externe est utilisée séparément pour évaluer les modèles candidats, sans intervenir dans leur entraînement ni dans leur sélection interne.

Afin de préserver la cohérence temporelle du protocole et de limiter les risques de fuite de données, le **bagging** et la **validation croisée** internes d'AutoGluon seront désactivés.

L'évaluation finale sera réalisée sur la partition de test commune aux différentes solutions AutoML.

#### 1. Definition de la configuration de l'expérience AutoGluon Tabular - Horizon 7 jours

| Élément | Configuration retenue |
|---|---|
| Type de problème | Régression |
| Variable cible | `target_volatility_7d` |
| Métrique de sélection | RMSE |
| Données d'entraînement | `train_7d` |
| Données de validation | `validation_7d` |
| Bagging | Désactivé (`num_bag_folds=0`) |
| Stacking | Désactivé (`num_stack_levels=0`) |
| Budget d'entraînement | 600 secondes |
| Ressources de calcul | CPU uniquement |
| Graine aléatoire | 42 |
| Jeu de test final | `test_7d` — réservé à l'évaluation finale |
| Purge temporelle | 8 observations, déjà appliquée aux partitions externes |


#### 2. Configuration proposée pour AutoGluon et entrainement

In [23]:
import psutil

print("Version psutil :", psutil.__version__)

Version psutil : 7.2.2


In [26]:
import psutil
import time
import threading

SEED = 42

# Fixer l'aléatoire du module random natif de Python
random.seed(SEED)

# Fixer l'aléatoire du module numpy
np.random.seed(SEED)

# Initialisation du predicteur
predictor_7d = TabularPredictor(
    label="target_volatility_7d",
    problem_type="regression",
    eval_metric="root_mean_squared_error",
    path=str(outputs_dir_7d)
)
# Dictionnaire pour stocker les mesures des ressources
resource_samples_7d = {
    "cpu_percent":[],
    "ram_used_mb":[]
}

# Événement pour arrêter la surveillance,
# des ressources pendant l'entraînement des modèles
stop_monitoring_7d = threading.Event()


def monitor_resources_7d():
    psutil.cpu_percent(interval=None)
    while not stop_monitoring_7d.is_set():
        resource_samples_7d["cpu_percent"].append(
            psutil.cpu_percent(interval=None)
        )
        resource_samples_7d["ram_used_mb"].append(
            psutil.virtual_memory().used / (1024**2)
        )
        stop_monitoring_7d.wait(0.2)

# Création et démarrage du thread de surveillance des ressources
monitoring_thread = threading.Thread(
    target=monitor_resources_7d,
    daemon=True
)

# Début du chronométrage 
# et démarrage du thread de surveillance des ressources
start_time_7d = time.perf_counter()
monitoring_thread.start()

try:
   predictor_7d.fit(
       train_data=train_7d_model,
       tuning_data=validation_7d_model,
       time_limit=600,
       num_bag_folds=0,
       num_stack_levels=0,
       auto_stack=False,
       num_gpus=0,
       fit_strategy="sequential",
       refit_full=False
   )
   
finally:
    stop_monitoring_7d.set()
    monitoring_thread.join()
    
# Mesurer les ressources    
training_time_7d = time.perf_counter() - start_time_7d

peak_cpu_7d = max(
    resource_samples_7d["cpu_percent"]
)

peak_ram_7d = max(
    resource_samples_7d["ram_used_mb"]
)

print("Temps d'entraînement :", round(training_time_7d, 2), "secondes")
print("Pic CPU :", round(peak_cpu_7d, 2), "%")
print("Pic RAM utilisée :", round(peak_ram_7d, 2), "Mo")


Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.3
Python Version:     3.12.9
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26100
CPU Count:          4
Pytorch Version:    2.13.0+cpu
CUDA Version:       CUDA is not available
Memory Avail:       9.43 GB / 23.89 GB (39.5%)
Disk Space Avail:   592.46 GB / 639.65 GB (92.6%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='extreme'  : Use this if you have a GPU. The go-to preset for best results, and the one to use for benchmark comparisons. New in v1.6: far better than 'best' on datasets <100000 samples by using Tabular Foundation Models (TFMs) meta-learned on https://tabarena.ai: Nori, TabICLv2, and TabDPT-Turbo. Every mod

Temps d'entraînement : 89.37 secondes
Pic CPU : 100.0 %
Pic RAM utilisée : 15156.62 Mo


#### 3. Valider et afficher les métriques via le leaderboard fourni pr AutoGluon

In [27]:
leaderboard_7d = predictor_7d.leaderboard(
    validation_7d_model,
    silent=True
)

display(
    leaderboard_7d[
        [
            "model",
            "score_val",
            "score_test",
            "fit_time",
            "pred_time_val"
        ]
    ]
)

,model,score_val,score_test,fit_time,pred_time_val
0,WeightedEnsemble_L2,-0.010943,-0.010943,37.674569,0.032519
1,NeuralNetTorch,-0.012053,-0.012053,34.396766,0.026749
2,LightGBMXT,-0.015257,-0.015257,3.230854,0.005771
3,CatBoost,-0.016556,-0.016556,3.211849,0.000000
4,LightGBMLarge,-0.017729,-0.017729,3.043669,0.004615
5,XGBoost,-0.017943,-0.017943,2.270865,0.007716
6,LightGBM,-0.018135,-0.018135,1.979709,0.004019
7,NeuralNetFastAI,-0.023129,-0.023129,15.244694,0.029080
8,ExtraTreesMSE,-0.031149,-0.031149,3.442417,0.178403
9,RandomForestMSE,-0.037108,-0.037108,9.574980,0.228174


#### 4. Calculer les métriques sur la validation

In [28]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import pandas as pd
import numpy as np

target_7d = "target_volatility_7d"

top_models_7d = [
    "WeightedEnsemble_L2",
    "NeuralNetTorch",
    "LightGBMXT"
]

X_val_7d = validation_7d_model.drop(columns=[target_7d])
y_val_7d = validation_7d_model[target_7d]

results_7d = []

for model_name in top_models_7d:

    y_pred = predictor_7d.predict(
        X_val_7d,
        model=model_name
    )

    results_7d.append({
        "model": model_name,
        "MAE": mean_absolute_error(y_val_7d, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_val_7d, y_pred)),
        "R2": r2_score(y_val_7d, y_pred)
    })

validation_results_7d = pd.DataFrame(results_7d)
display(validation_results_7d.sort_values("RMSE"))

,model,MAE,RMSE,R2
0,WeightedEnsemble_L2,0.008754,0.010943,-0.088537
1,NeuralNetTorch,0.009257,0.012053,-0.320505
2,LightGBMXT,0.013194,0.015257,-1.115976


### B. Configuration de l'expérience AutoGluon à 30 jours

L'expérience utilise exclusivement la partition d'entraînement préalablement figée pour l'apprentissage des modèles.

La partition de validation externe est utilisée séparément pour évaluer les modèles candidats, sans intervenir dans leur entraînement ni dans leur sélection interne.

Afin de préserver la cohérence temporelle du protocole et de limiter les risques de fuite de données, le **bagging** et la **validation croisée** internes d'AutoGluon seront désactivés.

L'évaluation finale sera réalisée sur la partition de test commune aux différentes solutions AutoML.

#### 1. Definition de la configuration de l'expérience AutoGluon Tabular - Horizon 30 jours

| Élément | Configuration retenue |
|---|---|
| Type de problème | Régression |
| Variable cible | `target_volatility_30d` |
| Métrique de sélection | RMSE |
| Données d'entraînement | `train_30d` |
| Données de validation | `validation_30d` |
| Bagging | Désactivé (`num_bag_folds=0`) |
| Stacking | Désactivé (`num_stack_levels=0`) |
| Budget d'entraînement | 600 secondes |
| Ressources de calcul | CPU uniquement |
| Graine aléatoire | 42 |
| Jeu de test final | `test_30d` — réservé à l'évaluation finale |
| Purge temporelle | 8 observations, déjà appliquée aux partitions externes |


#### 2. Configuration proposée pour AutoGluon et entrainement

In [29]:
import psutil
import time
import threading

SEED = 42

# Fixer l'aléatoire du module random natif de Python
random.seed(SEED)

# Fixer l'aléatoire du module numpy
np.random.seed(SEED)

# Initialisation du predicteur
predictor_30d = TabularPredictor(
    label="target_volatility_30d",
    problem_type="regression",
    eval_metric="root_mean_squared_error",
    path=str(outputs_dir_30d)
)
# Dictionnaire pour stocker les mesures des ressources
resource_samples_30d = {
    "cpu_percent":[],
    "ram_used_mb":[]
}

# Événement pour arrêter la surveillance,
# des ressources pendant l'entraînement des modèles
stop_monitoring_30d = threading.Event()


def monitor_resources_30d():
    psutil.cpu_percent(interval=None)
    while not stop_monitoring_30d.is_set():
        resource_samples_30d["cpu_percent"].append(
            psutil.cpu_percent(interval=None)
        )
        resource_samples_30d["ram_used_mb"].append(
            psutil.virtual_memory().used / (1024**2)
        )
        stop_monitoring_30d.wait(0.2)

# Création et démarrage du thread de surveillance des ressources
monitoring_thread = threading.Thread(
    target=monitor_resources_30d,
    daemon=True
)

# Début du chronométrage 
# et démarrage du thread de surveillance des ressources
start_time_30d = time.perf_counter()
monitoring_thread.start()

try:
   predictor_30d.fit(
       train_data=train_30d_model,
       tuning_data=validation_30d_model,
       time_limit=600,
       num_bag_folds=0,
       num_stack_levels=0,
       auto_stack=False,
       num_gpus=0,
       fit_strategy="sequential",
       refit_full=False
   )
   
finally:
    stop_monitoring_30d.set()
    monitoring_thread.join()
    
# Mesurer les ressources    
training_time_30d = time.perf_counter() - start_time_30d

peak_cpu_30d = max(
    resource_samples_30d["cpu_percent"]
)

peak_ram_30d = max(
    resource_samples_30d["ram_used_mb"]
)

print("Temps d'entraînement :", round(training_time_30d, 2), "secondes")
print("Pic CPU :", round(peak_cpu_30d, 2), "%")
print("Pic RAM utilisée :", round(peak_ram_30d, 2), "Mo")


Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.3
Python Version:     3.12.9
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26100
CPU Count:          4
Pytorch Version:    2.13.0+cpu
CUDA Version:       CUDA is not available
Memory Avail:       9.28 GB / 23.89 GB (38.8%)
Disk Space Avail:   592.24 GB / 639.65 GB (92.6%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='extreme'  : Use this if you have a GPU. The go-to preset for best results, and the one to use for benchmark comparisons. New in v1.6: far better than 'best' on datasets <100000 samples by using Tabular Foundation Models (TFMs) meta-learned on https://tabarena.ai: Nori, TabICLv2, and TabDPT-Turbo. Every mod

Temps d'entraînement : 44.1 secondes
Pic CPU : 100.0 %
Pic RAM utilisée : 15354.15 Mo


#### 3. Valider et afficher les métriques via le leaderboard fourni par AutoGluon

In [30]:
leaderboard_30d = predictor_30d.leaderboard(
    validation_30d_model,
    silent=True
)

display(
    leaderboard_30d[
        [
            "model",
            "score_val",
            "score_test",
            "fit_time",
            "pred_time_val"
        ]
    ]
)

,model,score_val,score_test,fit_time,pred_time_val
0,WeightedEnsemble_L2,-0.006839,-0.006839,11.496154,0.021173
1,NeuralNetTorch,-0.008719,-0.008719,9.513965,0.020175
2,CatBoost,-0.018493,-0.018493,1.930857,0.000000
3,LightGBMLarge,-0.019229,-0.019229,3.248775,0.003003
4,LightGBM,-0.019908,-0.019908,1.304133,0.005171
5,LightGBMXT,-0.020006,-0.020006,1.531916,0.004023
6,XGBoost,-0.021437,-0.021437,1.807224,0.008795
7,NeuralNetFastAI,-0.034480,-0.034480,9.531698,0.014728
8,ExtraTreesMSE,-0.038532,-0.038532,2.978666,0.235945
9,RandomForestMSE,-0.064963,-0.064963,9.101875,0.168626


#### 4. Calculer les métriques sur la validation

In [33]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import pandas as pd
import numpy as np

target_30d = "target_volatility_30d"

top_models_30d = [
    "WeightedEnsemble_L2",
    "NeuralNetTorch",
    "CatBoost"
]

X_val_30d = validation_30d_model.drop(columns=[target_30d])
y_val_30d = validation_30d_model[target_30d]

results_30d = []

for model_name in top_models_30d:

    y_pred = predictor_30d.predict(
        X_val_30d,
        model=model_name
    )

    results_30d.append({
        "model": model_name,
        "MAE": mean_absolute_error(y_val_30d, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_val_30d, y_pred)),
        "R2": r2_score(y_val_30d, y_pred)
    })

validation_results_30d = pd.DataFrame(results_30d)
display(validation_results_30d.sort_values("RMSE"))

,model,MAE,RMSE,R2
0,WeightedEnsemble_L2,0.005481,0.006839,-0.037630
1,NeuralNetTorch,0.006873,0.008719,-0.686338
2,CatBoost,0.017205,0.018493,-6.586244



## 6. Évaluation finale des modèles AutoGluon à 7 et 30 jours

### A. Évaluation finale - Horizon 7 jours

À l'issue de l'entraînement et de la validation, les trois modèles candidats retenus sont évalués sur la partition de test `test_7d`, préalablement figée dans le protocole commun.

Cette étape vise à mesurer leur capacité de généralisation sur des observations qui n'ont été utilisées ni pour l'entraînement ni pour la sélection des modèles.

L'évaluation repose sur les indicateurs suivants :

- **MAE** : erreur absolue moyenne ;
- **RMSE** : racine de l'erreur quadratique moyenne ;
- **R²** : coefficient de détermination ;
- **Temps d'inférence** : durée nécessaire à la génération des prédictions.

Aucun nouvel entraînement ni ajustement des modèles n'est effectué à cette étape.

### B. Évaluation finale - Horizon 30 jours

La même procédure est appliquée aux trois modèles candidats retenus pour l'horizon 30 jours, à partir de la partition figée `test_30d`.

Les métriques et les temps d'inférence sont calculés selon les mêmes modalités que pour l'horizon 7 jours.

### C. Conservation des résultats

Les résultats des deux horizons sont enregistrés afin d'alimenter le tableau comparatif final des solutions AutoML.

Leur interprétation et leur comparaison avec PyCaret seront réalisées dans un notebook distinct, consacré à l'analyse des performances et à la justification du choix technologique.


### A. Évaluer les trois modèles sur test_7d

In [37]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pandas as pd
import numpy as np
import time

target_7d = "target_volatility_7d"

# Partition de test figée
X_test_7d = test_7d_model.drop(columns=[target_7d])
y_test_7d = test_7d_model[target_7d]

# Modèles sélectionnés pendant la validation
top_models_7d = [
    "WeightedEnsemble_L2",
    "NeuralNetTorch",
    "LightGBMXT"
]

test_results_7d = []

for model_name in top_models_7d:

    start_time = time.perf_counter()

    y_pred = predictor_7d.predict(
        X_test_7d,
        model=model_name
    )

    inference_time = time.perf_counter() - start_time

    test_results_7d.append({
        "model": model_name,
        "MAE": mean_absolute_error(y_test_7d, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test_7d, y_pred)),
        "R2": r2_score(y_test_7d, y_pred),
        "inference_time_seconds": inference_time
    })

final_results_7d = pd.DataFrame(test_results_7d)

display(final_results_7d)

,model,MAE,RMSE,R2,inference_time_seconds
0,WeightedEnsemble_L2,0.008544,0.011058,-0.005152,0.159108
1,NeuralNetTorch,0.007820,0.011156,-0.023035,0.254139
2,LightGBMXT,0.013604,0.015526,-0.981313,0.018693


### B. Évaluer les trois modèles sur test_30d

In [38]:
target_30d = "target_volatility_30d"

X_test_30d = test_30d_model.drop(columns=[target_30d])
y_test_30d = test_30d_model[target_30d]

top_models_30d = [
    "WeightedEnsemble_L2",
    "NeuralNetTorch",
    "CatBoost"
]

test_results_30d = []

for model_name in top_models_30d:

    start_time = time.perf_counter()

    y_pred = predictor_30d.predict(
        X_test_30d,
        model=model_name
    )

    inference_time = time.perf_counter() - start_time

    test_results_30d.append({
        "model": model_name,
        "MAE": mean_absolute_error(y_test_30d, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test_30d, y_pred)),
        "R2": r2_score(y_test_30d, y_pred),
        "inference_time_seconds": inference_time
    })

final_results_30d = pd.DataFrame(test_results_30d)

display(final_results_30d)

,model,MAE,RMSE,R2,inference_time_seconds
0,WeightedEnsemble_L2,0.006251,0.007711,-0.065475,0.078105
1,NeuralNetTorch,0.005722,0.008374,-0.256412,0.097500
2,CatBoost,0.019382,0.020651,-6.642020,0.021151


### C. Exporter les résultats

In [40]:
from pathlib import Path

# Répertoire des résultats du benchmark
export_dir = project_root / "docs" / "benchmark" / "results_benchmark"
export_dir.mkdir(parents=True, exist_ok=True)

# Ajouter les informations nécessaires à la comparaison
export_7d = final_results_7d.copy()
export_7d.insert(0, "tool", "AutoGluon")
export_7d.insert(1, "horizon", "7d")

export_30d = final_results_30d.copy()
export_30d.insert(0, "tool", "AutoGluon")
export_30d.insert(1, "horizon", "30d")

# Regrouper les résultats des deux horizons
autogluon_results = pd.concat(
    [export_7d, export_30d],
    ignore_index=True
)

# Mesures globales des ressources pendant l'entraînement
resources_by_horizon = {
    "7d": {
        "training_time_seconds": round(training_time_7d, 2),
        "peak_cpu_percent": round(peak_cpu_7d, 2),
        "peak_ram_used_mb": round(peak_ram_7d, 2)
    },
    "30d": {
        "training_time_seconds": round(training_time_30d, 2),
        "peak_cpu_percent": round(peak_cpu_30d, 2),
        "peak_ram_used_mb": round(peak_ram_30d, 2)
    }
}

# Ajouter les mesures des ressources au tableau
for metric in [
    "training_time_seconds",
    "peak_cpu_percent",
    "peak_ram_used_mb"
]:
    autogluon_results[metric] = autogluon_results["horizon"].map(
        lambda h: resources_by_horizon[h][metric]
    )

# Export CSV
autogluon_results.to_csv(
    export_dir / "autogluon_test_results.csv",
    index=False
)

print("Export terminé :", export_dir / "autogluon_test_results.csv")
display(autogluon_results)

Export terminé : d:\ProjectFolderDevAI_2025-2026\Certification BLOCS 2&3\Projet-VolatiChainXplorerAI\docs\benchmark\results_benchmark\autogluon_test_results.csv


,tool,horizon,model,MAE,RMSE,R2,inference_time_seconds,training_time_seconds,peak_cpu_percent,peak_ram_used_mb
0,AutoGluon,7d,WeightedEnsemble_L2,0.008544,0.011058,-0.005152,0.159108,89.37,100.0,15156.62
1,AutoGluon,7d,NeuralNetTorch,0.007820,0.011156,-0.023035,0.254139,89.37,100.0,15156.62
2,AutoGluon,7d,LightGBMXT,0.013604,0.015526,-0.981313,0.018693,89.37,100.0,15156.62
3,AutoGluon,30d,WeightedEnsemble_L2,0.006251,0.007711,-0.065475,0.078105,44.10,100.0,15354.15
4,AutoGluon,30d,NeuralNetTorch,0.005722,0.008374,-0.256412,0.097500,44.10,100.0,15354.15
5,AutoGluon,30d,CatBoost,0.019382,0.020651,-6.642020,0.021151,44.10,100.0,15354.15
